
# MLP Cross-Regime Generalization Experiment

This notebook trains the MLP sequence-to-sequence surrogate on one formation regime and tests it on the other two regimes. It uses only the selected physics-guided feature set from the ablation study: current, terminal voltage, and square-root cumulative charge.

Experiment design:

- Train on protocols 1--4 from one regime
- Validate on protocol 5 from the same regime
- Test on protocol 6 from the two remaining regimes
- Input length = 50, prediction horizon = 50
- Training stride = 5, test stride = 50
- Model = MLP direct sequence-to-sequence surrogate


In [1]:

from pathlib import Path
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
try:
    print("MPS available:", torch.backends.mps.is_available())
except Exception:
    print("MPS available: False")


PyTorch version: 2.10.0
CUDA available: False
MPS available: True


In [ ]:

# Configuration

SEED = 63
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

INPUT_LENGTH = 50
PREDICTION_HORIZON = 50
TRAIN_STRIDE = 5
TEST_STRIDE = 50

BATCH_SIZE = 256
EPOCHS = 50
LEARNING_RATE = 1e-3
PATIENCE = 8
WEIGHT_DECAY = 1e-5

REGIMES = ["multi", "slow", "fast"]
TRAIN_PROTOCOLS = [1, 2, 3, 4]
VAL_PROTOCOLS = [5]
TEST_PROTOCOLS = [6]

# Selected feature set from the ablation result: baseline + sqrt cumulative charge
INPUT_COLS = [
    "Current [A]",
    "Terminal voltage [V]",
    "Sqrt cumulative charge [sqrt(A.s)]",
]

OUTPUT_COLS = [
    "SEI growth rate [nm/s]",
    "Negative SEI thickness [nm]",
    "Cell temperature [K]",
]

RAW_DIR = Path("../data/raw")
RESULTS_DIR = Path("../results_cross_regime")
MODELS_DIR = Path("../models_cross_regime")

if not RAW_DIR.exists():
    RAW_DIR = Path("data/raw")
    RESULTS_DIR = Path("results_cross_regime")
    MODELS_DIR = Path("models_cross_regime")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR.resolve())
print("RESULTS_DIR:", RESULTS_DIR.resolve())


Using device: mps
RAW_DIR: /Users/bikeshshr/Documents/battery_seq2seq_surrogate/data/raw
RESULTS_DIR: /Users/bikeshshr/Documents/battery_seq2seq_surrogate/results_cross_regime


In [ ]:

# Load raw CSV datasets

def load_dataset(regime, protocol_id):
    path = RAW_DIR / f"{regime}_{protocol_id}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset: {path}")
    df = pd.read_csv(path)
    return df

required_cols = set(INPUT_COLS + OUTPUT_COLS)

dataframes = {}
for regime in REGIMES:
    for pid in range(1, 7):
        df = load_dataset(regime, pid)
        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(f"{regime}_{pid}.csv is missing columns: {missing}")
        dataframes[(regime, pid)] = df

print("Loaded datasets:", len(dataframes))
for key, df in list(dataframes.items())[:3]:
    print(key, df.shape)


Loaded datasets: 18
('multi', 1) (155258, 11)
('multi', 2) (181653, 11)
('multi', 3) (185483, 11)


In [ ]:

# Standardization and windowing


def fit_standardizer(dfs, columns):
    values = np.vstack([df[columns].values.astype(np.float32) for df in dfs])
    mean = values.mean(axis=0).astype(np.float32)
    std = values.std(axis=0).astype(np.float32)
    std[std == 0] = 1.0
    return mean, std


def transform_array(values, mean, std):
    return (values.astype(np.float32) - mean) / std


def create_windows_from_dataframe(
    df,
    input_cols,
    output_cols,
    input_length,
    prediction_horizon,
    stride,
    x_mean,
    x_std,
    y_mean,
    y_std,
):
    x_raw = df[input_cols].values.astype(np.float32)
    y_raw = df[output_cols].values.astype(np.float32)

    x_scaled = transform_array(x_raw, x_mean, x_std)
    y_scaled = transform_array(y_raw, y_mean, y_std)

    X_list = []
    Y_list = []

    max_start = len(df) - input_length - prediction_horizon
    for start in range(0, max_start + 1, stride):
        input_start = start
        input_end = start + input_length
        output_start = input_end
        output_end = input_end + prediction_horizon
        X_list.append(x_scaled[input_start:input_end])
        Y_list.append(y_scaled[output_start:output_end])

    if len(X_list) == 0:
        return None, None

    return np.stack(X_list).astype(np.float32), np.stack(Y_list).astype(np.float32)


def make_package_for_train_regime(train_regime):
    train_dfs = [dataframes[(train_regime, pid)] for pid in TRAIN_PROTOCOLS]
    val_dfs = [dataframes[(train_regime, pid)] for pid in VAL_PROTOCOLS]

    x_mean, x_std = fit_standardizer(train_dfs, INPUT_COLS)
    y_mean, y_std = fit_standardizer(train_dfs, OUTPUT_COLS)

    X_train_list, Y_train_list = [], []
    for df in train_dfs:
        X, Y = create_windows_from_dataframe(
            df, INPUT_COLS, OUTPUT_COLS,
            INPUT_LENGTH, PREDICTION_HORIZON, TRAIN_STRIDE,
            x_mean, x_std, y_mean, y_std,
        )
        X_train_list.append(X)
        Y_train_list.append(Y)

    X_val_list, Y_val_list = [], []
    for df in val_dfs:
        X, Y = create_windows_from_dataframe(
            df, INPUT_COLS, OUTPUT_COLS,
            INPUT_LENGTH, PREDICTION_HORIZON, TRAIN_STRIDE,
            x_mean, x_std, y_mean, y_std,
        )
        X_val_list.append(X)
        Y_val_list.append(Y)

    X_train = np.concatenate(X_train_list, axis=0)
    Y_train = np.concatenate(Y_train_list, axis=0)
    X_val = np.concatenate(X_val_list, axis=0)
    Y_val = np.concatenate(Y_val_list, axis=0)

    return {
        "X_train": X_train,
        "Y_train": Y_train,
        "X_val": X_val,
        "Y_val": Y_val,
        "x_mean": x_mean,
        "x_std": x_std,
        "y_mean": y_mean,
        "y_std": y_std,
    }


def make_test_windows(test_regime, x_mean, x_std, y_mean, y_std):
    # Use protocol 6 from the target regime and non-overlapping stride H=50
    df = dataframes[(test_regime, 6)]
    X_test, Y_test = create_windows_from_dataframe(
        df, INPUT_COLS, OUTPUT_COLS,
        INPUT_LENGTH, PREDICTION_HORIZON, TEST_STRIDE,
        x_mean, x_std, y_mean, y_std,
    )
    return X_test, Y_test


In [ ]:

# Dataset, model, training, prediction

class Seq2SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


class MLPDirectSeq2Seq(nn.Module):
    def __init__(self, input_length, input_dim, hidden_dim, output_dim, horizon, dropout=0.2):
        super().__init__()
        self.horizon = horizon
        self.output_dim = output_dim
        self.network = nn.Sequential(
            nn.Linear(input_length * input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, horizon * output_dim),
        )

    def forward(self, x):
        batch_size = x.size(0)
        x = x.reshape(batch_size, -1)
        out = self.network(x)
        return out.view(batch_size, self.horizon, self.output_dim)


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    criterion = nn.MSELoss()
    total_loss = 0.0
    total_count = 0

    with torch.set_grad_enabled(is_train):
        for X, Y in loader:
            X = X.to(device)
            Y = Y.to(device)
            if is_train:
                optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, Y)
            if is_train:
                loss.backward()
                optimizer.step()
            batch_size = X.size(0)
            total_loss += loss.item() * batch_size
            total_count += batch_size

    return total_loss / total_count


def predict(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, Y in loader:
            X = X.to(device)
            pred = model(X).cpu().numpy()
            preds.append(pred)
            trues.append(Y.numpy())
    return np.concatenate(preds, axis=0), np.concatenate(trues, axis=0)


def inverse_transform_y(y_scaled, y_mean, y_std):
    return y_scaled * y_std.reshape(1, 1, -1) + y_mean.reshape(1, 1, -1)


def compute_metrics(y_true, y_pred, output_cols):
    rows = []
    eps = 1e-12
    for i, col in enumerate(output_cols):
        yt = y_true[:, :, i].reshape(-1)
        yp = y_pred[:, :, i].reshape(-1)
        err = yp - yt
        mse = np.mean(err ** 2)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(err))
        true_std = np.std(yt)
        nmae_std = mae / (true_std + eps)
        nrmse_std = rmse / (true_std + eps)
        ss_res = np.sum((yt - yp) ** 2)
        ss_tot = np.sum((yt - np.mean(yt)) ** 2)
        r2 = 1 - ss_res / (ss_tot + eps)
        rows.append({
            "output": col,
            "MSE": mse,
            "RMSE": rmse,
            "MAE": mae,
            "NMAE_std": nmae_std,
            "NRMSE_std": nrmse_std,
            "R2": r2,
        })
    return pd.DataFrame(rows)


In [ ]:


# Train one MLP per source regime and test on the other two regimes

all_metric_rows = []
summary_rows = []
training_summaries = []

for train_regime in REGIMES:
    print(" " + "=" * 80)
    print(f"Training on regime: {train_regime}")
    print("=" * 80)

    package = make_package_for_train_regime(train_regime)
    X_train, Y_train = package["X_train"], package["Y_train"]
    X_val, Y_val = package["X_val"], package["Y_val"]
    y_mean, y_std = package["y_mean"], package["y_std"]

    train_loader = DataLoader(Seq2SeqDataset(X_train, Y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(Seq2SeqDataset(X_val, Y_val), batch_size=BATCH_SIZE, shuffle=False)

    model = MLPDirectSeq2Seq(
        input_length=INPUT_LENGTH,
        input_dim=len(INPUT_COLS),
        hidden_dim=256,
        output_dim=len(OUTPUT_COLS),
        horizon=PREDICTION_HORIZON,
        dropout=0.2,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    start_time = time.time()

    for epoch in range(1, EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer=optimizer)
        val_loss = run_epoch(model, val_loader, optimizer=None)
        scheduler.step(val_loss)
        print(f"Epoch {epoch:03d} | train={train_loss:.6f} | val={val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    training_time = time.time() - start_time
    model.load_state_dict(best_state)
    model.to(device)

    model_path = MODELS_DIR / f"mlp_sqrtcc_train_{train_regime}_best.pt"
    torch.save({
        "model_state_dict": model.state_dict(),
        "train_regime": train_regime,
        "input_cols": INPUT_COLS,
        "output_cols": OUTPUT_COLS,
        "best_val_loss": best_val_loss,
        "x_mean": package["x_mean"],
        "x_std": package["x_std"],
        "y_mean": package["y_mean"],
        "y_std": package["y_std"],
    }, model_path)

    training_summaries.append({
        "train_regime": train_regime,
        "train_windows": len(X_train),
        "val_windows": len(X_val),
        "best_val_loss": best_val_loss,
        "training_time_s": training_time,
        "model_path": str(model_path),
    })

    for test_regime in [r for r in REGIMES if r != train_regime]:
        print(f"Testing train={train_regime} -> test={test_regime}")
        X_test, Y_test = make_test_windows(
            test_regime,
            package["x_mean"], package["x_std"],
            package["y_mean"], package["y_std"],
        )
        test_loader = DataLoader(Seq2SeqDataset(X_test, Y_test), batch_size=BATCH_SIZE, shuffle=False)

        pred_scaled, true_scaled = predict(model, test_loader)
        pred = inverse_transform_y(pred_scaled, y_mean, y_std)
        true = inverse_transform_y(true_scaled, y_mean, y_std)

        metrics = compute_metrics(true, pred, OUTPUT_COLS)
        metrics.insert(0, "train_regime", train_regime)
        metrics.insert(1, "test_regime", test_regime)
        metrics.insert(2, "test_windows", len(X_test))
        all_metric_rows.append(metrics)

        row = {"train_regime": train_regime, "test_regime": test_regime, "test_windows": len(X_test)}
        for _, m in metrics.iterrows():
            out = m["output"]
            if "growth" in out:
                prefix = "growth"
            elif "thickness" in out:
                prefix = "thickness"
            elif "temperature" in out:
                prefix = "temperature"
            else:
                prefix = out
            row[f"{prefix}_NMAE"] = m["NMAE_std"]
            row[f"{prefix}_NRMSE"] = m["NRMSE_std"]
            row[f"{prefix}_R2"] = m["R2"]
        summary_rows.append(row)

combined_metrics = pd.concat(all_metric_rows, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)
training_summary_df = pd.DataFrame(training_summaries)

metrics_path = RESULTS_DIR / "mlp_sqrtcc_cross_regime_metrics_long.csv"
summary_path = RESULTS_DIR / "mlp_sqrtcc_cross_regime_summary.csv"
training_path = RESULTS_DIR / "mlp_sqrtcc_cross_regime_training_summary.csv"

combined_metrics.to_csv(metrics_path, index=False)
summary_df.to_csv(summary_path, index=False)
training_summary_df.to_csv(training_path, index=False)

print("Saved:")
print(metrics_path.resolve())
print(summary_path.resolve())
print(training_path.resolve())

display(summary_df)


Training on regime: multi
Epoch 001 | train=0.077810 | val=0.026637
Epoch 002 | train=0.034785 | val=0.021242
Epoch 003 | train=0.031080 | val=0.013820
Epoch 004 | train=0.028867 | val=0.015348
Epoch 005 | train=0.027281 | val=0.022477
Epoch 006 | train=0.026628 | val=0.018255
Epoch 007 | train=0.025790 | val=0.018150
Epoch 008 | train=0.022806 | val=0.018664
Epoch 009 | train=0.022593 | val=0.023893
Epoch 010 | train=0.021966 | val=0.016309
Epoch 011 | train=0.021602 | val=0.020245
Early stopping at epoch 11.
Testing train=multi -> test=slow
Testing train=multi -> test=fast
Training on regime: slow
Epoch 001 | train=0.090775 | val=0.043430
Epoch 002 | train=0.027959 | val=0.019999
Epoch 003 | train=0.022580 | val=0.018667
Epoch 004 | train=0.019739 | val=0.016429
Epoch 005 | train=0.017912 | val=0.015109
Epoch 006 | train=0.017144 | val=0.017156
Epoch 007 | train=0.016802 | val=0.010725
Epoch 008 | train=0.016596 | val=0.017506
Epoch 009 | train=0.015854 | val=0.014883
Epoch 010 | tra

,train_regime,test_regime,test_windows,growth_NMAE,growth_NRMSE,growth_R2,thickness_NMAE,thickness_NRMSE,thickness_R2,temperature_NMAE,temperature_NRMSE,temperature_R2
0,multi,slow,2316,0.312779,0.332103,0.889707,0.408414,0.503624,0.746362,0.235103,0.354458,0.874360
1,multi,fast,242,1.080371,1.365909,-0.865708,9.580075,10.408479,-107.336433,0.337082,0.477625,0.771874
2,slow,multi,2347,0.258861,0.444893,0.802070,0.280429,0.403043,0.837557,0.268785,0.383909,0.852614
3,slow,fast,242,0.362088,0.489355,0.760532,7.591403,9.161053,-82.924889,0.323792,0.450505,0.797045
4,fast,multi,2347,1.077667,1.391586,-0.936511,0.836964,0.986186,0.027436,1.879496,2.478188,-5.141417
5,fast,slow,2316,2.354527,2.589877,-5.707464,1.374588,1.538663,-1.367483,3.337581,3.769619,-13.210027


In [ ]:

# Create compact tuple table for the paper: (NMAE, NRMSE, R2)

def fmt_tuple(nmae, nrmse, r2):
    return f"({nmae:.3f}, {nrmse:.3f}, {r2:.3f})"

paper_rows = []
for _, row in summary_df.iterrows():
    paper_rows.append({
        "Train Regime": row["train_regime"].capitalize(),
        "Test Regime": row["test_regime"].capitalize(),
        "SEI Growth Rate": fmt_tuple(row["growth_NMAE"], row["growth_NRMSE"], row["growth_R2"]),
        "SEI Thickness": fmt_tuple(row["thickness_NMAE"], row["thickness_NRMSE"], row["thickness_R2"]),
        "Temperature": fmt_tuple(row["temperature_NMAE"], row["temperature_NRMSE"], row["temperature_R2"]),
    })

paper_table_df = pd.DataFrame(paper_rows)
paper_table_path = RESULTS_DIR / "mlp_sqrtcc_cross_regime_paper_table.csv"
paper_table_df.to_csv(paper_table_path, index=False)

print("Saved paper table:", paper_table_path.resolve())
display(paper_table_df)


Saved paper table: /Users/bikeshshr/Documents/battery_seq2seq_surrogate/results_cross_regime/mlp_sqrtcc_cross_regime_paper_table.csv


,Train Regime,Test Regime,SEI Growth Rate,SEI Thickness,Temperature
0,Multi,Slow,"(0.313, 0.332, 0.890)","(0.408, 0.504, 0.746)","(0.235, 0.354, 0.874)"
1,Multi,Fast,"(1.080, 1.366, -0.866)","(9.580, 10.408, -107.336)","(0.337, 0.478, 0.772)"
2,Slow,Multi,"(0.259, 0.445, 0.802)","(0.280, 0.403, 0.838)","(0.269, 0.384, 0.853)"
3,Slow,Fast,"(0.362, 0.489, 0.761)","(7.591, 9.161, -82.925)","(0.324, 0.451, 0.797)"
4,Fast,Multi,"(1.078, 1.392, -0.937)","(0.837, 0.986, 0.027)","(1.879, 2.478, -5.141)"
5,Fast,Slow,"(2.355, 2.590, -5.707)","(1.375, 1.539, -1.367)","(3.338, 3.770, -13.210)"


In [ ]:

# Generate LaTeX table rows for quick copy/paste

print("LaTeX rows:")
for _, row in paper_table_df.iterrows():
    print(
        f"{row['Train Regime']} & {row['Test Regime']} & "
        f"{row['SEI Growth Rate']} & {row['SEI Thickness']} & {row['Temperature']} \\\\"
    )


LaTeX rows:
Multi & Slow & (0.313, 0.332, 0.890) & (0.408, 0.504, 0.746) & (0.235, 0.354, 0.874) \\
Multi & Fast & (1.080, 1.366, -0.866) & (9.580, 10.408, -107.336) & (0.337, 0.478, 0.772) \\
Slow & Multi & (0.259, 0.445, 0.802) & (0.280, 0.403, 0.838) & (0.269, 0.384, 0.853) \\
Slow & Fast & (0.362, 0.489, 0.761) & (7.591, 9.161, -82.925) & (0.324, 0.451, 0.797) \\
Fast & Multi & (1.078, 1.392, -0.937) & (0.837, 0.986, 0.027) & (1.879, 2.478, -5.141) \\
Fast & Slow & (2.355, 2.590, -5.707) & (1.375, 1.539, -1.367) & (3.338, 3.770, -13.210) \\
